# Sistema de Gestão e Análise de Vendas

## Análise do e-commerce brasileiro com dados públicos da Olist

Projeto de portfólio — Pedro Vinícius

Esta análise utiliza o **Brazilian E-Commerce Public Dataset by Olist**. A base é pública, anonimizada e composta por arquivos relacionais de pedidos, itens, produtos, clientes, vendedores, pagamentos, avaliações e localização.

> Os resultados apresentados nesta versão são calculados a partir dos dados públicos da Olist fornecidos para este projeto.

## 1. Objetivo

Construir uma análise reprodutível de vendas com foco em quatro perguntas: **quanto foi vendido, quando as vendas ocorreram, onde a receita se concentrou e quais categorias e clientes se destacaram**.

A análise mantém a granularidade de item de pedido e evita joins diretos de tabelas com cardinalidade 1:N que poderiam duplicar métricas.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("../dados/raw")

files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

orders = pd.read_csv(BASE_DIR / files["orders"])
items = pd.read_csv(BASE_DIR / files["items"])
customers = pd.read_csv(BASE_DIR / files["customers"])
products = pd.read_csv(BASE_DIR / files["products"])
sellers = pd.read_csv(BASE_DIR / files["sellers"])
translation = pd.read_csv(BASE_DIR / files["translation"])

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

sales = (
    items
    .merge(orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp"]], on="order_id", how="left", validate="many_to_one")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left", validate="many_to_one")
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left", validate="many_to_one")
    .merge(translation, on="product_category_name", how="left", validate="many_to_one")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one")
]
sales["item_revenue"] = sales["price"]
sales["item_plus_freight"] = sales["price"] + sales["freight_value"]
sales["is_realized_sale"] = sales["order_status"].eq("delivered")
sales["purchase_month"] = sales["order_purchase_timestamp"].dt.to_period("M").astype(str)

realized = sales.loc[sales["is_realized_sale"]].copy()
realized.head()

## 2. Indicadores principais

Para evitar ambiguidade, **receita** nesta análise significa a soma de `price` dos itens pertencentes a pedidos com status `delivered`. Frete é apresentado separadamente.

In [ ]:
kpis = {
    "Pedidos entregues": int(realized["order_id"].nunique()),
    "Itens entregues": int(len(realized)),
    "Clientes únicos": int(realized["customer_unique_id"].nunique()),
    "Receita de itens (R$)": float(realized["price"].sum()),
    "Frete (R$)": float(realized["freight_value"].sum()),
    "Receita + frete (R$)": float(realized["price"].sum() + realized["freight_value"].sum()),
    "Ticket médio por pedido (R$)": float(realized["price"].sum() / realized["order_id"].nunique()),
}

pd.Series(kpis, name="valor")

## 3. Evolução mensal

A série mensal permite identificar crescimento, concentração temporal e possíveis períodos de pico. O último período observado deve ser interpretado com cautela quando houver poucos dias disponíveis na fonte.

In [ ]:
monthly = (
    realized.groupby("purchase_month", as_index=False)
    .agg(revenue=("price", "sum"), orders=("order_id", "nunique"), items=("order_item_id", "size"))
)
monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["purchase_month"], monthly["revenue"], marker="o")
ax.set_title("Evolução mensal da receita de itens")
ax.set_xlabel("Mês")
ax.set_ylabel("Receita (R$)")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## 4. Receita por estado do cliente

A localização é analisada pelo estado do cliente. Isso responde à concentração geográfica da demanda sem confundir estado do cliente com estado do vendedor.

In [ ]:
state_revenue = (
    realized.groupby("customer_state", as_index=False)
    .agg(revenue=("price", "sum"), orders=("order_id", "nunique"), items=("order_item_id", "size"))
    .sort_values("revenue", ascending=False)
)
state_revenue.head(10)

In [ ]:
top_states = state_revenue.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_states["customer_state"], top_states["revenue"])
ax.set_title("Top 10 estados por receita de itens")
ax.set_xlabel("UF")
ax.set_ylabel("Receita (R$)")
plt.tight_layout()
plt.show()

## 5. Categorias de produtos

Categorias são avaliadas por receita e quantidade de itens. A comparação evita concluir que uma categoria líder em receita também é líder em volume.

In [ ]:
category_revenue = (
    realized.groupby("product_category_name_english", dropna=False, as_index=False)
    .agg(revenue=("price", "sum"), items=("order_item_id", "size"))
    .sort_values("revenue", ascending=False)
)
category_revenue.head(15)

In [ ]:
top_categories = category_revenue.dropna(subset=["product_category_name_english"]).head(10)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_categories["product_category_name_english"][::-1], top_categories["revenue"][::-1])
ax.set_title("Top 10 categorias por receita de itens")
ax.set_xlabel("Receita (R$)")
ax.set_ylabel("Categoria")
plt.tight_layout()
plt.show()

## 6. Recorrência de clientes

A recorrência é calculada sobre `customer_unique_id`, que representa o cliente ao longo de diferentes pedidos. O indicador mede clientes com mais de um pedido entre os pedidos entregues.

In [ ]:
customer_orders = realized.groupby("customer_unique_id")["order_id"].nunique()
recurring_customers = int((customer_orders > 1).sum())
total_customers = int(customer_orders.size)
recurrence_rate = recurring_customers / total_customers

pd.Series({
    "Clientes únicos": total_customers,
    "Clientes com mais de um pedido": recurring_customers,
    "Taxa de recorrência": recurrence_rate,
}, name="valor")

## 7. Limitações e cuidados de interpretação

- O dataset é histórico e não representa necessariamente o mercado brasileiro atual.
- `price` e `freight_value` são métricas diferentes; frete não é tratado como receita de produto.
- Pedidos não entregues permanecem disponíveis na base operacional, mas não entram no KPI de receita realizada definido neste projeto.
- Pagamentos e avaliações não são agregados diretamente à fato de itens sem uma etapa específica de controle de cardinalidade.
- Categorias sem tradução para inglês permanecem como ausentes na coluna traduzida; isso não implica que o produto não tenha categoria na origem.